# Подготовка модели распознавания рукописных букв и цифр

## Imports

In [32]:
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
from torch import nn
import torch
from PIL import Image

### DataPreparation

In [ ]:
dataset = EMNIST('data/', 'balanced', download=False)

In [33]:
train_transform = transforms.Compose([
    transforms.RandomRotation(10),        # случайный поворот ±10°
    transforms.RandomAffine(0, translate=(0.1, 0.1)),  # случайный сдвиг
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # для теста только нормализация
])

In [ ]:
len(dataset)

112800

In [34]:
train_dataset = EMNIST('data/', 'balanced', train=True, download=True, transform=train_transform)
val_dataset = EMNIST('data/', 'balanced', train=False, download=False, transform=test_transform)

In [35]:
print(len(train_dataset))
print(len(val_dataset))

112800
18800


In [36]:
train_loader = DataLoader(train_dataset, batch_size=1000, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1000)

In [37]:
n_classes = len(train_dataset.classes)
print(n_classes)

47


In [38]:
for i in range(10):
    img, lbl = train_dataset[i]
    img = torch.rot90(img, 1, [1, 2])
    print(lbl, img.shape)
    display(transforms.functional.to_pil_image(img))

45 torch.Size([1, 28, 28])


36 torch.Size([1, 28, 28])


43 torch.Size([1, 28, 28])


15 torch.Size([1, 28, 28])


4 torch.Size([1, 28, 28])


42 torch.Size([1, 28, 28])


26 torch.Size([1, 28, 28])


32 torch.Size([1, 28, 28])


20 torch.Size([1, 28, 28])


1 torch.Size([1, 28, 28])


### Modeling

Для начала планирую протестировать обычную логрег

#### LogReg

In [ ]:
class LogReg(nn.Module):
    def __init__(self, in_features, n_classes):
        super(LogReg, self).__init__()
        self.fc = nn.Linear(in_features, n_classes)

    def forward(self, x):
        return self.fc(x)

In [ ]:
model = LogReg(in_features=28*28, n_classes=n_classes)
loss_f = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=1e-1)

n_epoch = 50
val_fre = 10

model.train()
for epoch in range(n_epoch):
    loss_sum = 0
    for step, (data, target) in enumerate(train_loader):
        data = data.flatten(start_dim=1) #разворачиваю в одномерный вектор
        optimizer.zero_grad()
        output = model(data)
        loss = loss_f(output, target)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f'Epoch: {epoch} \tLoss: {loss_sum / (step + 1):.6f}')

    if (epoch + 1) % val_fre == 0:
        model.eval()
        loss_sum = 0
        correct = 0
        for step, (data, target) in enumerate(val_loader):
            data = data.flatten(start_dim=1)
            with torch.no_grad():
                output = model(data)
                loss = loss_f(output, target)
            loss_sum += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
        acc = correct / len(val_loader.dataset)
        print(f'Val Loss: {loss_sum / (step + 1):.6f} \tAccuracy:{acc}')
        model.train()

C:\Anaconda\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch: 0 	Loss: 2.050498
Epoch: 1 	Loss: 1.442406
Epoch: 2 	Loss: 1.330538
Epoch: 3 	Loss: 1.274435
Epoch: 4 	Loss: 1.239089
Epoch: 5 	Loss: 1.214929
Epoch: 6 	Loss: 1.196396
Epoch: 7 	Loss: 1.181076
Epoch: 8 	Loss: 1.170241
Epoch: 9 	Loss: 1.159401
Val Loss: 1.196200 	Accuracy:0.6742553191489362
Epoch: 10 	Loss: 1.152592
Epoch: 11 	Loss: 1.145577
Epoch: 12 	Loss: 1.139073
Epoch: 13 	Loss: 1.133823
Epoch: 14 	Loss: 1.128369
Epoch: 15 	Loss: 1.124099
Epoch: 16 	Loss: 1.120910
Epoch: 17 	Loss: 1.116451
Epoch: 18 	Loss: 1.113529
Epoch: 19 	Loss: 1.110686
Val Loss: 1.169356 	Accuracy:0.678031914893617
Epoch: 20 	Loss: 1.107473
Epoch: 21 	Loss: 1.105166
Epoch: 22 	Loss: 1.102713
Epoch: 23 	Loss: 1.099951
Epoch: 24 	Loss: 1.098233
Epoch: 25 	Loss: 1.096276
Epoch: 26 	Loss: 1.094521
Epoch: 27 	Loss: 1.092588
Epoch: 28 	Loss: 1.091463
Epoch: 29 	Loss: 1.090077
Val Loss: 1.150891 	Accuracy:0.6885106382978723
Epoch: 30 	Loss: 1.087807
Epoch: 31 	Loss: 1.086751
Epoch: 32 	Loss: 1.084778
Epoch: 33

ВЫВОД: точность модели 0,687 - 0,688 улучшать модель не буду, попробую mlp

####

#### MLP v1.0

Буду использовать GPU поэтому явно укажу это методом .to(device)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [39]:
def train(model, optimizer, loss_f, train_loader, val_loader, n_epoch, val_fre):
    model.train()
    for epoch in range(n_epoch):
        loss_sum = 0
        for step, (data, target) in enumerate(train_loader):
            data = data.flatten(start_dim=1).to(device) #разворачиваю в одномерный вектор
            target = target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = loss_f(output, target)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

        print(f'Epoch: {epoch} \tLoss: {loss_sum / (step + 1):.6f}')

        if (epoch + 1) % val_fre == 0:
            validate(model, val_loader)

In [17]:
def validate(model, val_loader):
    model.eval()
    loss_sum = 0
    correct = 0
    for step, (data, target) in enumerate(val_loader):
        data = data.flatten(start_dim=1).to(device)
        target = target.to(device)
        with torch.no_grad():
            output = model(data)
            loss = loss_f(output, target)
        loss_sum += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
    acc = correct / len(val_loader.dataset)
    print(f'Val Loss: {loss_sum / (step + 1):.6f} \tAccuracy:{acc}')
    model.train()

In [18]:
class MLP(nn.Module):
    def __init__(self, in_features, hid_features, n_classes, dropout=0.3):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_features, hid_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hid_features, n_classes)
        self.dropout = nn.Dropout(dropout)
        self.bn1 = nn.BatchNorm1d(hid_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [19]:
model = MLP(in_features=28*28, hid_features=512, n_classes=n_classes).to(device)

loss_f = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epoch = 100
val_fre = 10

In [20]:
train(model, optimizer, loss_f, train_loader, val_loader, n_epoch, val_fre)

Epoch: 0 	Loss: 1.376293
Epoch: 1 	Loss: 0.851423
Epoch: 2 	Loss: 0.718829
Epoch: 3 	Loss: 0.641914
Epoch: 4 	Loss: 0.593830
Epoch: 5 	Loss: 0.556301
Epoch: 6 	Loss: 0.529678
Epoch: 7 	Loss: 0.506058
Epoch: 8 	Loss: 0.486200
Epoch: 9 	Loss: 0.470608
Val Loss: 0.515884 	Accuracy:0.8306914893617021
Epoch: 10 	Loss: 0.454524
Epoch: 11 	Loss: 0.444744
Epoch: 12 	Loss: 0.431323
Epoch: 13 	Loss: 0.422474
Epoch: 14 	Loss: 0.414516
Epoch: 15 	Loss: 0.404700
Epoch: 16 	Loss: 0.397153
Epoch: 17 	Loss: 0.388230
Epoch: 18 	Loss: 0.382062
Epoch: 19 	Loss: 0.374120
Val Loss: 0.472600 	Accuracy:0.8457978723404256
Epoch: 20 	Loss: 0.367806
Epoch: 21 	Loss: 0.364380
Epoch: 22 	Loss: 0.359481
Epoch: 23 	Loss: 0.352589
Epoch: 24 	Loss: 0.349267
Epoch: 25 	Loss: 0.341612
Epoch: 26 	Loss: 0.338336
Epoch: 27 	Loss: 0.333771
Epoch: 28 	Loss: 0.331068
Epoch: 29 	Loss: 0.326103
Val Loss: 0.471298 	Accuracy:0.8487765957446809
Epoch: 30 	Loss: 0.319550
Epoch: 31 	Loss: 0.319263
Epoch: 32 	Loss: 0.313742
Epoch: 3

Началось переобучение после 30 эпохи

#### MLP V2.0

Добавляю scheduler.step() для динамичного изменения learning rate + меня стохастический оптимайзер на Adam

Также добавляю дополнительный слой

In [ ]:
def train(model, optimizer, loss_f, train_loader, val_loader, n_epoch, val_fre):
    model.train()
    for epoch in range(n_epoch):
        loss_sum = 0
        for step, (data, target) in enumerate(train_loader):
            data = data.flatten(start_dim=1).to(device) #разворачиваю в одномерный вектор
            target = target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = loss_f(output, target)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

        print(f'Epoch: {epoch} \tLoss: {loss_sum / (step + 1):.6f}')

        if (epoch + 1) % val_fre == 0:
            validate(model, val_loader)

In [40]:
def validate(model, val_loader):
    model.eval()
    loss_sum = 0
    correct = 0
    for step, (data, target) in enumerate(val_loader):
        data = data.flatten(start_dim=1).to(device)
        target = target.to(device)
        with torch.no_grad():
            output = model(data)
            loss = loss_f(output, target)
        loss_sum += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
    scheduler.step(loss_sum)
    acc = correct / len(val_loader.dataset)
    print(f'Val Loss: {loss_sum / (step + 1):.6f} \tAccuracy:{acc}')
    model.train()


In [41]:
class MLP(nn.Module):
    def __init__(self, in_features, hid_features, n_classes, dropout=0.5):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_features, hid_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hid_features, hid_features // 2)
        self.dropout = nn.Dropout(dropout)
        self.bn1 = nn.BatchNorm1d(hid_features)
        self.bn2 = nn.BatchNorm1d(hid_features // 2)
        self.fc3 = nn.Linear(hid_features // 2, n_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MLP(in_features=28*28, hid_features=512, n_classes=n_classes).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

loss_f = nn.CrossEntropyLoss()

n_epoch = 100
val_fre = 10

In [43]:
train(model, optimizer, loss_f, train_loader, val_loader, n_epoch, val_fre)

Epoch: 0 	Loss: 2.439962
Epoch: 1 	Loss: 1.569393
Epoch: 2 	Loss: 1.314022
Epoch: 3 	Loss: 1.193181
Epoch: 4 	Loss: 1.112669
Epoch: 5 	Loss: 1.062366
Epoch: 6 	Loss: 1.015570
Epoch: 7 	Loss: 0.989492
Epoch: 8 	Loss: 0.958149
Epoch: 9 	Loss: 0.941254
Val Loss: 0.513830 	Accuracy:0.8287765957446809
Epoch: 10 	Loss: 0.915838
Epoch: 11 	Loss: 0.898044
Epoch: 12 	Loss: 0.881548
Epoch: 13 	Loss: 0.876131
Epoch: 14 	Loss: 0.858889
Epoch: 15 	Loss: 0.852221
Epoch: 16 	Loss: 0.845858
Epoch: 17 	Loss: 0.841424
Epoch: 18 	Loss: 0.832545
Epoch: 19 	Loss: 0.820484
Val Loss: 0.462820 	Accuracy:0.8425531914893617
Epoch: 20 	Loss: 0.815365
Epoch: 21 	Loss: 0.808291
Epoch: 22 	Loss: 0.807259
Epoch: 23 	Loss: 0.800149
Epoch: 24 	Loss: 0.793664
Epoch: 25 	Loss: 0.791237
Epoch: 26 	Loss: 0.792834
Epoch: 27 	Loss: 0.787440
Epoch: 28 	Loss: 0.782437
Epoch: 29 	Loss: 0.778168
Val Loss: 0.438200 	Accuracy:0.8486170212765958
Epoch: 30 	Loss: 0.773116
Epoch: 31 	Loss: 0.770794
Epoch: 32 	Loss: 0.769141
Epoch: 3

KeyboardInterrupt: 

Получилось улучшить показатели на 1,6% это значительно но хотелось бы приблизиться к 90% поэтому буду использовать другую модель

#### CNN v1.0

Буду использовать сверточный метод с 2 ядрами 3х3

In [44]:
class CNN(nn.Module):
    def __init__(self, n_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 5 * 5, n_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.flatten(start_dim=1)
        x = self.fc(x)
        return x

Необходимо убрать преобразование в вектор в функциях

Необходимо убрать преобразование в вектор

In [45]:
def validate(model, val_loader):
    model.eval()
    loss_sum = 0
    correct = 0
    for step, (data, target) in enumerate(val_loader):
        data = data.to(device)  # убрал flatten
        target = target.to(device)
        with torch.no_grad():
            output = model(data)
            loss = loss_f(output, target)
        loss_sum += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
    scheduler.step(loss_sum)
    acc = correct / len(val_loader.dataset)
    print(f'Val Loss: {loss_sum / (step + 1):.6f} \tAccuracy:{acc}')
    model.train()

def train(model, optimizer, loss_f, train_loader, val_loader, n_epoch, val_fre):
    model.train()
    for epoch in range(n_epoch):
        loss_sum = 0
        for step, (data, target) in enumerate(train_loader):
            data = data.to(device)  # убрал flatten
            target = target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = loss_f(output, target)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        print(f'Epoch: {epoch} \tLoss: {loss_sum / (step + 1):.6f}')
        if (epoch + 1) % val_fre == 0:
            validate(model, val_loader)

In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_cnn = CNN(n_classes=n_classes).to(device)

optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn, 'min', patience=5)

loss_f = nn.CrossEntropyLoss()

n_epoch = 50
val_fre = 10

In [47]:
train(model_cnn, optimizer_cnn, loss_f, train_loader, val_loader, n_epoch, val_fre)

Epoch: 0 	Loss: 1.730031
Epoch: 1 	Loss: 0.894600
Epoch: 2 	Loss: 0.771979
Epoch: 3 	Loss: 0.704004
Epoch: 4 	Loss: 0.651352
Epoch: 5 	Loss: 0.610260
Epoch: 6 	Loss: 0.581517
Epoch: 7 	Loss: 0.555463
Epoch: 8 	Loss: 0.528961
Epoch: 9 	Loss: 0.515727
Val Loss: 0.408495 	Accuracy:0.861968085106383
Epoch: 10 	Loss: 0.501293
Epoch: 11 	Loss: 0.484254
Epoch: 12 	Loss: 0.478175
Epoch: 13 	Loss: 0.466518
Epoch: 14 	Loss: 0.457595
Epoch: 15 	Loss: 0.449506
Epoch: 16 	Loss: 0.445807
Epoch: 17 	Loss: 0.436245
Epoch: 18 	Loss: 0.433104
Epoch: 19 	Loss: 0.428961
Val Loss: 0.367796 	Accuracy:0.874095744680851
Epoch: 20 	Loss: 0.422649
Epoch: 21 	Loss: 0.418356
Epoch: 22 	Loss: 0.415011
Epoch: 23 	Loss: 0.411980
Epoch: 24 	Loss: 0.405592
Epoch: 25 	Loss: 0.403674
Epoch: 26 	Loss: 0.400504
Epoch: 27 	Loss: 0.397805
Epoch: 28 	Loss: 0.395738
Epoch: 29 	Loss: 0.391235
Val Loss: 0.345003 	Accuracy:0.8811702127659574
Epoch: 30 	Loss: 0.390693
Epoch: 31 	Loss: 0.388889
Epoch: 32 	Loss: 0.384704
Epoch: 33 

Результат получилось улучшить еще на 2.2%

### Итоги

In [50]:
validate(model_cnn, val_loader)

Val Loss: 0.333487 	Accuracy:0.8827127659574469


In [49]:
torch.save(model.state_dict(), 'model.ckpt')

Принято решение остановиться на таком результате. Были опробованы 3 вида моделей для решения задачи, лучше всего себя показала CNN для будущих тестов стоит попробовать поменять размер ядра и количество иттераций для обработки(возможно увеличить количество эпох). 

Также для улучшения попробовать: 

1. ансамбли моделей
2. Готовые решения моделей
3. добавить в mlp 4 слой
4. добавить больше шума в тренировочные данные